In [ ]:
import json
import re
import torch
from bs4 import BeautifulSoup
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# ===================== LOAD DATA =====================
with open("/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/train data/vlsp_2025_train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

with open("/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/law_db/vlsp2025_law.json", "r") as f:
    data_law = json.load(f)

# ===================== CLEAN TEXT =====================
def clean_html(text):
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

def remove_special_tags(text):
    text = re.sub(r"<<IMAGE:.*?>>", " ", text)

    tables = re.findall(r"<<TABLE:(.*?)>>", text, flags=re.DOTALL)
    table_text = ""
    for table in tables:
        table_text += " " + BeautifulSoup(table, "html.parser").get_text(separator=" ")

    text = re.sub(r"<<TABLE:.*?>>", " ", text, flags=re.DOTALL)
    return text + " " + table_text

def normalize(text):
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    return normalize(clean_html(remove_special_tags(text)))

# clean law
for law in data_law:
    for article in law["articles"]:
        if "text" in article and article["text"]:
            article["text"] = clean_text(article["text"])

# mapping
data_law_mapping = {law["id"]: law["articles"] for law in data_law}

# ===================== FORMAT =====================
def get_law_text(sample):
    laws = []
    for art in sample.get("relevant_articles", []):
        try:
            idx = int(art["article_id"]) - 1
            laws.append(data_law_mapping[art["law_id"]][idx]["text"])
        except:
            continue
    return " ".join(laws)

def format_sample(sample):
    q = sample["question"]
    answer = sample["answer"]
    law_text = get_law_text(sample)

    if "choices" in sample:
        c = sample["choices"]
        input_text = f"""Luật: {law_text}

Câu hỏi: {q}

A. {c['A']}
B. {c['B']}
C. {c['C']}
D. {c['D']}"""

        output_text = f"""Đáp án: {answer}.
Giải thích: {c[answer]}"""
    else:
        input_text = f"""Luật: {law_text}

Câu hỏi: {q}"""
        output_text = answer

    return {
        "instruction": "Trả lời câu hỏi luật giao thông",
        "input": input_text,
        "output": output_text
    }

formatted = [format_sample(x) for x in data]

# ===================== SPLIT =====================
train_data, val_data = train_test_split(formatted, test_size=0.1, random_state=42)

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

In [ ]:
#============

In [ ]:
# ===================== TOKENIZER =====================
model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def tokenize(batch):
    prompts = []

    for i in range(len(batch["instruction"])):
        prompt = f"""### Instruction:
{batch['instruction'][i]}

### Input:
{batch['input'][i]}

### Response:
{batch['output'][i]}"""
        prompts.append(prompt)

    tokens = tokenizer(
        prompts,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# remove text columns
train_dataset = train_dataset.remove_columns(["instruction", "input", "output"])
val_dataset = val_dataset.remove_columns(["instruction", "input", "output"])

# ===================== MODEL =====================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ===================== TRAINING =====================
training_args = TrainingArguments(
    output_dir="./qwen-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    remove_unused_columns=False,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

In [ ]:
def infer(question):
    prompt = f"""### Instruction:
Trả lời câu hỏi luật giao thông

### Input:
Câu hỏi: {question}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# test
print(infer("Vạch trắng gồm 2 dải đứt đoạn và thẳng liền song song nhau là gì?"))

In [ ]:
#trainer.save_model("./qwen-lora-final")

In [ ]:
#!zip -r qwen-lora.zip ./qwen-lora

In [ ]:
#!zip -r qwen-lora-final.zip ./qwen-lora-final

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ===================== CONFIG =====================
base_model_name = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/kaggle/input/datasets/sntngphm/qwen-lora/qwen-lora-final"


# ===================== LOAD TOKENIZER =====================
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# ===================== LOAD MODEL =====================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

In [ ]:
test_path = "/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/public_test data/vlsp_2025_public_test.json"
law_path = "/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/law_db/vlsp2025_law.json"


# ===================== LOAD DATA =====================
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

with open(law_path, "r", encoding="utf-8") as f:
    law_data = json.load(f)

# mapping law
law_mapping = {}
for law in law_data:
    law_mapping[law["id"]] = law["articles"]

In [ ]:
def get_law_text(sample):
    texts = []
    for art in sample.get("relevant_articles", []):
        try:
            law_id = art["law_id"]
            idx = int(art["article_id"]) - 1
            texts.append(law_mapping[law_id][idx]["text"])
        except:
            continue
    return " ".join(texts)

In [ ]:
def infer(sample):
    law_text = get_law_text(sample)
    law_text = law_text[:2000]  # 🔥 cắt bớt trước

    q = sample["question"]

    if "choices" in sample:
        c = sample["choices"]
        input_text = f"""Luật: {law_text}

Câu hỏi: {q}

A. {c['A']}
B. {c['B']}
C. {c['C']}
D. {c['D']}"""
    else:
        input_text = f"""Luật: {law_text}

Câu hỏi: {q}"""

    prompt = f"""### Instruction:
Trả lời câu hỏi luật giao thông

### Input:
{input_text}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024   # 🔥 cực quan trọng
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,   # giảm thêm cho chắc
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in result:
        result = result.split("### Response:")[-1].strip()

    return result

In [ ]:
correct = 0
total = 0

for sample in test_data:
    pred = infer(sample)
    gt = sample.get("answer")

    if not gt:
        continue

    # normalize
    pred = pred.strip().lower() if pred else ""
    gt = gt.strip().lower()

    if gt in pred:
        correct += 1
    total += 1
    print("=====" + total + "=====")
    print

accuracy = correct / total if total > 0 else 0
print(f"Accuracy: {accuracy:.4f} ({correct}/{total})")

In [ ]:
#=========

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece accelerate pillow

In [ ]:
import json
import re
from bs4 import BeautifulSoup

LAW_PATH = "/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/law_db/vlsp2025_law.json"

with open(LAW_PATH, "r") as f:
    data_law = json.load(f)

def clean_html(text):
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

def normalize(text):
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    return normalize(clean_html(text))

# clean law
for law in data_law:
    for article in law["articles"]:
        if "text" in article and article["text"]:
            article["text"] = clean_text(article["text"])

data_law_mapping = {law["id"]: law["articles"] for law in data_law}

def get_law_text(sample):
    laws = []
    for art in sample.get("relevant_articles", []):
        try:
            idx = int(art["article_id"]) - 1
            laws.append(data_law_mapping[art["law_id"]][idx]["text"])
        except:
            continue
    return " ".join(laws)[:1200]

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import torch

model_checkpoint = "OpenGVLab/InternVL3-1B-hf"

device = "cuda" if torch.cuda.is_available() else "cpu"

intern_processor = AutoProcessor.from_pretrained(model_checkpoint)
intern_model = AutoModelForImageTextToText.from_pretrained(
    model_checkpoint,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

vision_cache = {}

def describe_image(img_path):
    if img_path in vision_cache:
        return vision_cache[img_path]

    image = Image.open(img_path).convert("RGB").resize((448, 448))

    question = """
Quan sát ảnh biển báo giao thông tại Việt Nam.

Yêu cầu:
1. Xác định TẤT CẢ các biển báo xuất hiện trong ảnh (có thể có nhiều).
2. Với mỗi biển, mô tả NGẮN GỌN theo format:

[Biển i]
- loại: (biển cấm / biển nguy hiểm / biển chỉ dẫn / khác)
- hình dạng: (tròn / tam giác / vuông / chữ nhật)
- màu nền:
- màu viền:
- ký hiệu chính: (ví dụ: 1 gạch chéo, 2 gạch chéo, mũi tên, số,...)
- chữ/số (nếu có):

3. Không giải thích. Không suy đoán tên biển.

Chỉ mô tả những gì nhìn thấy trong ảnh.
"""

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    prompt = intern_processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = intern_processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )

    inputs = {k: v.to(intern_model.device) for k, v in inputs.items()}

    outputs = intern_model.generate(
        **inputs,
        max_new_tokens=200
    )

    response = intern_processor.batch_decode(
        outputs, skip_special_tokens=True
    )[0]

    vision_cache[img_path] = response
    return response

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_name = "Qwen/Qwen2.5-7B-Instruct"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)

# quantization (giữ lại nếu cần nhẹ VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# model (KHÔNG load LoRA)
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.eval()

In [ ]:
def clean_vision_text(text):
    # bỏ "user", "assistant"
    text = re.sub(r"user\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"assistant\s*", "", text, flags=re.IGNORECASE)

    # optional: cắt phần "Giải thích chi tiết"
    text = text.split("### Giải thích")[0]

    return text.strip()

def is_true_false(sample):
    q = sample.get("question", "").lower()

    # keyword phổ biến
    keywords = ["đúng hay sai", "đúng hoặc sai", "có đúng không", "phải không"]

    return any(k in q for k in keywords)

def build_prompt(sample, law_text, vision_text):
    q = sample.get("question", "")
    c = sample.get("choices") or sample.get("options")

    # ===== TRUE / FALSE =====
    if is_true_false(sample):
        return f"""
Bạn là chuyên gia luật giao thông Việt Nam.

Mô tả biển báo:
{vision_text}

Quy tắc:
{law_text}

Câu hỏi:
{q}

Chỉ trả lời: ĐÚNG hoặc SAI.
"""

    # ===== MCQ =====
    if c is None:
        return f"""
Bạn là chuyên gia luật giao thông Việt Nam.

Mô tả biển báo:
{vision_text}

Quy tắc:
{law_text}

Câu hỏi:
{q}

Chỉ trả lời 1 ký tự: A/B/C/D.
"""

    if isinstance(c, list):
        c = dict(zip(["A", "B", "C", "D"], c))

    if not isinstance(c, dict):
        c = {}

    return f"""
Bạn là chuyên gia luật giao thông Việt Nam.

Mô tả biển báo:
{vision_text}

Quy tắc:
{law_text}

Câu hỏi:
{q}

A. {c.get('A', '')}
B. {c.get('B', '')}
C. {c.get('C', '')}
D. {c.get('D', '')}

Chỉ trả lời 1 ký tự: A/B/C/D.
"""
    
import re

def solve_with_qwen(sample, law_text, vision_text):
    prompt = build_prompt(sample, law_text, vision_text)

    # 🔥 DEBUG
    print("\n================ PROMPT ================\n")
    print(prompt)

    print("\n================ VISION ================\n")
    print(vision_text)

    print("\n================ LAW ================\n")
    print(law_text)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False
        )

    output_ids = outputs[0][inputs.input_ids.shape[1]:]
    result = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    print("\n================ OUTPUT ================\n")
    print(result)

    # parse
    match = re.search(r"^[ABCD]$", result)
    if match:
        return match.group(0)

    match = re.search(r"\b([ABCD])\b", result)
    return match.group(1) if match else "A"

In [ ]:
import json
import re

TEST_PATH = "/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/public_test data/vlsp_2025_public_test.json"

IMAGE_DIR = "/kaggle/input/datasets/sntngphm/nlp-ds/VLSP2025-MLQA-TSR-main/dataset/public_test data/public_test_images/public_test_images"

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)


# ================= CLEAN VISION =================
def clean_vision_text(text):
    if text is None:
        return ""

    # bỏ role chat
    text = re.sub(r"\b(user|assistant)\b\s*", "", text, flags=re.IGNORECASE)

    # cắt phần giải thích dài
    text = text.split("### Giải thích")[0]

    # remove khoảng trắng thừa
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


# ================= INFER =================
from PIL import Image
import matplotlib.pyplot as plt

def infer(sample, debug=False):
    img_path = f"{IMAGE_DIR}/{sample['image_id']}.jpg"

    # load ảnh
    image = Image.open(img_path).convert("RGB")

    vision_text = describe_image(img_path)
    vision_text = clean_vision_text(vision_text)

    law_text = get_law_text(sample)

    if debug:
        print("\n========== DEBUG Q", sample['id'] ,"==========")

        # 🖼️ HIỆN ẢNH
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"Image ID: {sample['image_id']}")
        plt.show()

        print("VISION:\n", vision_text[:500])
        print("\nLAW:\n", law_text[:500])

    pred = solve_with_qwen(sample, law_text, vision_text)
    return pred


# ================= TEST LOOP =================
for i, sample in enumerate(test_data[:5]):  # test 5 sample trước
    pred = infer(sample, debug=True)

    print("\n" + "="*50)
    print("Q:", sample["question"])
    print("GT:", sample.get("answer", "N/A"))
    print("PRED:", pred)

In [ ]:
infer(test_data[1])

In [ ]:
results = []

for sample in test_data:
    pred = infer(sample)
    results.append({
        "id": sample["id"],
        "answer": pred
    })

with open("submission.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)